# Point & Ask prompt eval

Runs the classify -> score -> branch pipeline (`backend.point_and_ask`) against the synthetic fixture images in `eval/point_and_ask_images/`, then eyeballs explain-branch output quality on a couple of legitimate cases. Outputs are saved in this notebook on run.

In [1]:
import json
import os
import sys
from pathlib import Path

if not (Path.cwd() / "pyproject.toml").exists():
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

from backend.point_and_ask import classify_image, decide_branch, explain_image, score_risk

CASES_PATH = Path.cwd() / "eval" / "point_and_ask_eval_cases.json"
cases = json.loads(CASES_PATH.read_text())
len(cases)

10

In [2]:
passed = 0
for i, case in enumerate(cases, start=1):
    image_bytes = (Path.cwd() / "eval" / case["image"]).read_bytes()
    result = classify_image(image_bytes, "image/png")
    risk_level = score_risk(result)
    classification = decide_branch(result, risk_level)

    ok = classification == case["expected_classification"]
    passed += ok
    status = "PASS" if ok else "FAIL"
    print(
        f"[{status}] case {i} ({case['image']}): got={classification!r} "
        f"(risk={risk_level!r}) expected={case['expected_classification']!r}"
    )
print(f"\n{passed}/{len(cases)} passed")

2026-07-29 21:48:06.176 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


[PASS] case 1 (point_and_ask_images/case_01_scam_bank.png): got='scam' (risk='high') expected='scam'


[PASS] case 2 (point_and_ask_images/case_02_scam_prize.png): got='scam' (risk='medium') expected='scam'


[PASS] case 3 (point_and_ask_images/case_03_scam_tax.png): got='scam' (risk='high') expected='scam'


[PASS] case 4 (point_and_ask_images/case_04_legit_checkin.png): got='explain' (risk='low') expected='explain'


...

[PASS] case 6 (point_and_ask_images/case_07_scam_parcel.png): got='scam' (risk='medium') expected='scam'


[PASS] case 7 (point_and_ask_images/case_08_legit_medical.png): got='explain' (risk='low') expected='explain'


[PASS] case 8 (point_and_ask_images/case_09_scam_grandchild.png): got='scam' (risk='high') expected='scam'


[PASS] case 9 (point_and_ask_images/case_10_legit_community.png): got='explain' (risk='low') expected='explain'


[PASS] case 10 (point_and_ask_images/case_06_unreadable.png): got='unclear' (risk='low') expected='unclear'

10/10 passed


## Explain-branch quality check

Not pass/fail — eyeballing the actual explanation + translation text for a couple of legitimate documents, since the automated eval above only checks routing correctness.

In [3]:
test_cases = [
    ("case_05_legit_passport.png", "English"),
    ("case_05_legit_passport.png", "Mandarin Chinese"),
    ("case_08_legit_medical.png", "Malay"),
    ("case_08_legit_medical.png", "Tamil"),
]

for image_name, target_language in test_cases:
    image_bytes = (Path.cwd() / "eval" / "point_and_ask_images" / image_name).read_bytes()
    explanation = explain_image(image_bytes, "image/png", target_language)
    print(f"=== {image_name} -> {target_language} ===")
    print(explanation)
    print("-" * 60)

=== case_05_legit_passport.png -> English ===
This is a simple reminder notice about your passport.

**What it says:**
Your passport is going to expire next month. It's asking you to go renew it before that happens, at your nearest ICA office (that's the government office that handles passports and ID cards).

**What this means for you:**
Once a passport expires, you can't use it anymore — so it's important to get a new one before the expiry date if you plan to travel, or even just to have a valid ID.

**Suggestion:**
Since this involves a deadline, it might be a good idea to ask a family member to help you:
- Check the exact expiry date on your passport
- Find out what documents you need to bring
- Maybe go with you to the ICA office, or help you book an appointment if needed

Would you like help thinking of what questions to ask your family member about this?
------------------------------------------------------------


=== case_05_legit_passport.png -> Mandarin Chinese ===
您好！我来帮您看看这张通知写的是什么。

**这是一份提醒您的通知：**

📌 您的**护照下个月就要到期了**。

📌 通知建议您在护照过期之前，去**最近的ICA（移民与关卡局）办事处**办理**更新手续**。

---

**简单来说：** 您的护照快过期了，需要尽快去更换新的。

**小建议：** 护照更新涉及证件和可能的费用，办理时间也要提前安排。建议您和家人商量一下，请他们帮您确认具体的办理地点、需要带的材料，或者陪您一起去办理，这样会更顺利、更安心。

如果您需要，我也可以帮您列出一些常见需要准备的东西，但具体流程还是要以ICA官方说明或家人的确认为准哦。
------------------------------------------------------------


=== case_08_legit_medical.png -> Malay ===
Baik, saya terangkan surat ini ya.

Ini adalah **peringatan untuk temujanji doktor**.

📅 **Butiran temujanji:**
- Dengan siapa: Dr. Lim
- Hari: **Selasa**
- Masa: **3 petang (3pm)**
- Tempat: **Raffles Medical**

📌 **Apa yang perlu dibawa:**
- Kad pengenalan (NRIC) — jangan lupa bawa sekali ya.

**Cadangan saya:**
- Cuba tandakan tarikh Selasa ini di kalendar atau minta ahli keluarga tolong ingatkan sehari sebelum, supaya tidak terlepas.
- Kalau boleh, minta anak atau cucu teman pergi ke Raffles Medical, terutama kalau tempat itu agak jauh atau perlu tunggu lama.
- Sediakan NRIC awal-awal supaya tidak tergesa-gesa pada hari temujanji.

Kalau ada sebarang bayaran atau borang yang perlu ditandatangani semasa di sana, elok minta ahli keluarga teman sekali untuk membantu.
------------------------------------------------------------


=== case_08_legit_medical.png -> Tamil ===
இந்த காகிதம் ஒரு டாக்டர் அப்பாயின்ட்மென்ட் (சந்திப்பு) நினைவூட்டல் கடிதம்.

இதில் இருப்பது:

- **யாரிடம் செல்ல வேண்டும்:** டாக்டர் Lim
- **எப்போது:** செவ்வாய்க்கிழமை (Tuesday), மதியம் 3 மணிக்கு
- **எங்கே:** Raffles Medical (மருத்துவமனை/கிளினிக்)
- **என்ன கொண்டு வர வேண்டும்:** உங்கள் NRIC (அடையாள அட்டை)

எளிமையாக சொன்னால் - இது வரும் செவ்வாய்க்கிழமை மதியம் 3 மணிக்கு Raffles Medical-ல் டாக்டரை பார்க்க வேண்டும் என்பதை நினைவூட்டுகிறது. போகும்போது உங்கள் அடையாள அட்டையை (NRIC) மறக்காமல் எடுத்துச் செல்ல வேண்டும்.

💡 **ஒரு சிறு ஆலோசனை:** இந்த நாள் மற்றும் நேரத்தை உங்கள் குடும்பத்தினரிடம் ஒருமுறை சொல்லிவையுங்கள், அவர்கள் உங்களுக்கு நினைவூட்டவும் உதவலாம், தேவைப்பட்டால் அழைத்துச் செல்லவும் ஏற்பாடு செய்யலாம்.
------------------------------------------------------------
